# Preparacion de ambiente
Se crean las external locations (con la credencial de Managed Identity), el catalogo, los schemas y las tablas de cada capa.
Todas las sentencias usan `IF NOT EXISTS`, por lo que el notebook se puede ejecutar varias veces.

In [ ]:
%run ./00_utils

In [ ]:
params = obtener_parametros()
dbutils.widgets.text("storage_credential", "cred_instacart_jhon_garcia_15_nov")

CATALOGO = params["catalogo"]
STORAGE = params["storage"]
CREDENCIAL = dbutils.widgets.get("storage_credential")
log(f"catalogo={CATALOGO} | storage={STORAGE} | credential={CREDENCIAL}", entidad="prep_ambiente")

## External locations

In [ ]:
for contenedor in ["raw", "bronze", "silver", "golden", "catalog"]:
    try:
        spark.sql(f"""
            CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt_{contenedor}`
            URL '{ruta_contenedor(STORAGE, contenedor)}/'
            WITH (STORAGE CREDENTIAL `{CREDENCIAL}`)
            COMMENT 'Contenedor {contenedor}'
        """)
        log(f"exlt_{contenedor} creada", entidad="prep_ambiente")
    except Exception as e:
        log(f"No se pudo crear exlt_{contenedor}: {e}", "ERROR", "prep_ambiente")
        raise

## Catalogo y schemas

In [ ]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOGO}
    MANAGED LOCATION '{ruta_contenedor(STORAGE, 'catalog')}/{CATALOGO}'
""")

for capa in CAPAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{capa}")
log("Catalogo y schemas creados", entidad="prep_ambiente")

## Tablas

In [ ]:
AUDITORIA = "ingestion_date TIMESTAMP, source_file STRING"

TABLAS = {
    # bronze
    ("bronze", "orders"): f"""
        order_id INT, user_id INT, eval_set STRING, order_number INT, order_dow INT,
        order_hour_of_day INT, days_since_prior_order DOUBLE, {AUDITORIA}""",
    ("bronze", "order_products"): f"""
        order_id INT, product_id INT, add_to_cart_order INT, reordered INT,
        eval_set STRING, {AUDITORIA}""",
    ("bronze", "products"): f"""
        product_id INT, product_name STRING, aisle_id INT, department_id INT, {AUDITORIA}""",
    ("bronze", "aisles"): f"aisle_id INT, aisle STRING, {AUDITORIA}",
    ("bronze", "departments"): f"department_id INT, department STRING, {AUDITORIA}",

    # silver
    ("silver", "productos"): """
        product_id INT, nombre_producto STRING, aisle_id INT, pasillo STRING,
        department_id INT, departamento STRING, es_organico BOOLEAN, fecha_proceso TIMESTAMP""",
    ("silver", "ordenes"): """
        order_id INT, user_id INT, eval_set STRING, numero_orden INT, dia_semana_num INT,
        dia_semana STRING, hora_dia INT, franja_horaria STRING, dias_desde_orden_previa INT,
        es_primera_orden BOOLEAN, fecha_proceso TIMESTAMP""",
    ("silver", "detalle_ordenes"): """
        order_id INT, product_id INT, user_id INT, orden_en_carrito INT, es_recompra BOOLEAN,
        nombre_producto STRING, pasillo STRING, departamento STRING, es_organico BOOLEAN,
        numero_orden INT, dia_semana_num INT, dia_semana STRING, hora_dia INT, franja_horaria STRING,
        fecha_proceso TIMESTAMP""",

    # golden
    ("golden", "kpi_departamentos"): """
        departamento STRING, total_productos_vendidos BIGINT, total_ordenes BIGINT,
        total_clientes BIGINT, productos_distintos BIGINT, tasa_recompra DOUBLE,
        participacion_pct DOUBLE, fecha_proceso TIMESTAMP""",
    ("golden", "top_productos"): """
        departamento STRING, ranking INT, nombre_producto STRING, veces_comprado BIGINT,
        tasa_recompra DOUBLE, fecha_proceso TIMESTAMP""",
    ("golden", "actividad_dia_hora"): """
        dia_semana_num INT, dia_semana STRING, hora_dia INT, franja_horaria STRING,
        total_ordenes BIGINT, total_productos BIGINT, promedio_productos_orden DOUBLE,
        fecha_proceso TIMESTAMP""",
    ("golden", "segmentos_clientes"): """
        user_id INT, total_ordenes BIGINT, total_productos BIGINT, promedio_productos_orden DOUBLE,
        promedio_dias_entre_ordenes DOUBLE, tasa_recompra DOUBLE, departamento_favorito STRING,
        segmento STRING, fecha_proceso TIMESTAMP""",
}

PARTICIONES = {("silver", "detalle_ordenes"): "departamento"}

In [ ]:
for (capa, tabla), columnas in TABLAS.items():
    nombre = f"{CATALOGO}.{capa}.{tabla}"
    particion = f"PARTITIONED BY ({PARTICIONES[(capa, tabla)]})" if (capa, tabla) in PARTICIONES else ""
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {nombre} ({columnas})
        USING DELTA
        {particion}
        LOCATION '{ruta_tabla(STORAGE, capa, CATALOGO, tabla)}'
    """)
    log("Tabla creada", entidad=nombre)

In [ ]:
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.bronze")
        .unionByName(spark.sql(f"SHOW TABLES IN {CATALOGO}.silver"))
        .unionByName(spark.sql(f"SHOW TABLES IN {CATALOGO}.golden")))